# Clamped beam with the common MORFE API

This conservative St. Venant–Kirchhoff example uses only the public, physics-independent MORFE workflow. The committed output uses order 3. To reproduce the order-9 reference, change `order = 3` to `order = 9` and rerun the notebook.

Before the first run, run `julia setup.jl` in the bash from this repository. For more information look at the README.md of the repository.

`SVK` is only a short name for the structural backend used to describe the mechanical case; `build_model` and `parametrise` belong to the common API.

In [ ]:
using MORFE, MORFEFerrite
const SVK = MORFEFerrite.StructuralSVK # for Saint-Venant-Kirchhoff hyperelasticity

## 1. Describe the mechanical case

Choose the polynomial expansion order directly: `3` is the quick demonstration and `9` is the conservative reference calculation.

`mechanical_model` reads the mesh and supplies the physical information required by the structural backend. Here the Rayleigh damping coefficients are zero, so the beam is conservative. `dirichlet = "Dirichlet"` names a physical facet group stored in the Gmsh mesh; every displacement component on those labeled facets is fixed to zero, producing the clamped ends. The finite-element and quadrature orders use their API defaults.

In [ ]:
order = 3  # Change to 9 for the reference calculation.
case = SVK.mechanical_model(joinpath(@__DIR__, "clamped_clamped_beam.msh");
    material = SVK.SVKMaterial(E = 160e3, ν = 0.22, ρ = 2.32e-3),
    damping = SVK.RayleighDamping(α = 0.0, β = 0.0),
    dirichlet = "Dirichlet")

## 2. Build the MORFE model

`build_model` converts the structural case into MORFE's physics-independent model and computes its spectral data. `master = [1]` selects the first vibration-mode pair as the tangent space of the reduced model. The returned `meta` contains auxiliary backend information, including the selected eigenvalues displayed below.

In [ ]:
(; model, spectral, meta) = build_model(case; master = [1], expansion_order = order)
meta.spectrum.eigenvalues[meta.master_indices] # print master eigenvalues

## 3. Parametrise the invariant manifold

`parametrise` computes the polynomial manifold map `W` and its reduced dynamics `R` up to the chosen order. The resonance configuration keeps near-resonant monomials in complex normal form. Evaluating `R` at the end of the cell displays the reduced system.

In [ ]:
W, R = parametrise(model, spectral, order;
    resonance = ResonanceConfig(style = :complex_normal_form, tol = 0.05))
R # print reduced dynamics

## Optional: save the standard ROM files

The common saver writes `W`, `R`, their coefficient table, and a summary to the example's `results` directory. This cell is optional; the ROM is already available in memory after the previous cell.

In [ ]:
MORFE.save_rom(joinpath(@__DIR__, "results"), W, R);